In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the new dataset (Update the filename to match your file)
df7 = pd.read_csv("Detailed_Polling_Data.csv")

# Standardise column spacing and strip whitespace to prevent key errors
df7.columns = df7.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

# 2. Select the primary political party columns for analysis based on your exact schema
core_parties = [
    'All India Anna Dravida Munnetra Kazhagam', 
    'Bahujan Samaj Party', 
    'Naam Tamilar Katchi', 
    'Indian National Congress',
    'Tamilaga Vettri Kazhagam'
]

# Clean missing numerical fields by filling with 0
df7[core_parties] = df7[core_parties].fillna(0)

# Exact structural column definitions from your dataset
station_col = 'Serial No. Of Polling Station'
building_col = 'Location and Name of Building in Which Polling Station Located'
area_col = 'Polling Areas'

# Identify alternative/independent candidate columns dynamically if any are present
known_meta_cols = core_parties + [
    station_col, building_col, area_col, 'Total of Valid Votes', 
    'No. Of Rejected Votes', 'NOTA', 'Total', 'No. Of Tendered Votes', 'Category'
]
independent_candidate_cols = [c for c in df7.columns if c not in known_meta_cols]

# Calculate total independent votes dynamically
if independent_candidate_cols:
    df7['Total_Independent_Votes'] = df7[independent_candidate_cols].sum(axis=1)
else:
    df7['Total_Independent_Votes'] = 0

# 3. Calculate true total votes for normalization (Core + Independents + NOTA)
df7['Total_Calculated_Votes'] = df7[core_parties].sum(axis=1) + df7['Total_Independent_Votes'].fillna(0) + df7['NOTA'].fillna(0)

# Filter out empty entries to completely avoid division by zero errors
df7 = df7[df7['Total_Calculated_Votes'] > 0].copy()

# 4. Feature Engineering: Create normalized percentage shares (%)
share_cols = []
for party in core_parties:
    # Build a clean short column name for readability in summaries
    party_label = party.split()[-1] if len(party.split()) > 1 else party
    col_name = f'{party_label}_share_pct'
    df7[col_name] = (df7[party] / df7['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

# Append strategic structural dimensions
df7['independent_share_pct'] = (df7['Total_Independent_Votes'].fillna(0) / df7['Total_Calculated_Votes']) * 100

# Fallback check for missing analytical metrics from earlier script layers
if 'Margin_Percentage' not in df7.columns:
    if 'Winner_Votes' not in df7.columns:
        df7['Winner_Votes'] = df7[core_parties].max(axis=1)
        sorted_votes = np.sort(df7[core_parties].values, axis=1)
        df7['Runner_Up_Votes'] = sorted_votes[:, -2]
        df7['Margin_Of_Victory'] = df7['Winner_Votes'] - df7['Runner_Up_Votes']
        df7['Winner_Party'] = df7[core_parties].idxmax(axis=1)
    df7['Margin_Percentage'] = (df7['Margin_Of_Victory'] / df7['Total_Calculated_Votes']) * 100

feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']

# Drop or fill edge-case missing numbers inside target features
df7[feature_cols] = df7[feature_cols].fillna(0)

# 5. Extract and Scale features for the ML model
X = df7[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df7['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Raw Profile Breakdown to help map the text identities
print("\n--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df7.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 7: BOOTH COUNT PER CLUSTER ---")
print(df7['Cluster_ID'].value_counts())

# 8. Export individual target files for campaign ground teams
for cluster_num in range(optimal_k):
    target_cols = [station_col, building_col, area_col, 'Winner_Party', 'Margin_Percentage']
    # Filter only existing tracking columns to prevent target slice export errors
    valid_target_cols = [c for c in target_cols if c in df7.columns]
    
    cluster_df = df7[df7['Cluster_ID'] == cluster_num][valid_target_cols]
    filename = f"Dataset_7_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated for all 4 clusters.")


/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


TypeError: can only concatenate str (not "int") to str

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the new dataset (Update the filename to match your file)
df7 = pd.read_csv("Detailed_Polling_Data.csv")

# Standardise column spacing and strip whitespace to prevent key errors
df7.columns = df7.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

# 2. Select the primary political party columns for analysis based on your exact schema
core_parties = [
    'All India Anna Dravida Munnetra Kazhagam', 
    'Bahujan Samaj Party', 
    'Naam Tamilar Katchi', 
    'Indian National Congress',
    'Tamilaga Vettri Kazhagam'
]

# Clean missing numerical fields by filling with 0
df7[core_parties] = df7[core_parties].fillna(0)

# Exact structural column definitions from your dataset
station_col = 'Serial No. Of Polling Station'
building_col = 'Location and Name of Building in Which Polling Station Located'
area_col = 'Polling Areas'

# 3. Calculate true total votes for normalization (Core Parties + NOTA)
df7['Total_Calculated_Votes'] = df7[core_parties].sum(axis=1) + df7['NOTA'].fillna(0)

# Filter out empty entries to completely avoid division by zero errors
df7 = df7[df7['Total_Calculated_Votes'] > 0].copy()

# 4. Feature Engineering: Create normalized percentage shares (%)
share_cols = []
for party in core_parties:
    # Build a clean short column name for readability in summaries
    party_label = party.split()[-1] if len(party.split()) > 1 else party
    col_name = f'{party_label}_share_pct'
    df7[col_name] = (df7[party] / df7['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

# Fallback check for missing analytical metrics from earlier script layers
if 'Margin_Percentage' not in df7.columns:
    if 'Winner_Votes' not in df7.columns:
        df7['Winner_Votes'] = df7[core_parties].max(axis=1)
        sorted_votes = np.sort(df7[core_parties].values, axis=1)
        df7['Runner_Up_Votes'] = sorted_votes[:, -2]
        df7['Margin_Of_Victory'] = df7['Winner_Votes'] - df7['Runner_Up_Votes']
        df7['Winner_Party'] = df7[core_parties].idxmax(axis=1)
    df7['Margin_Percentage'] = (df7['Margin_Of_Victory'] / df7['Total_Calculated_Votes']) * 100

# Set up clean target feature tracking array without any independent metrics
feature_cols = share_cols + ['Margin_Percentage']

# Drop or fill edge-case missing numbers inside target features
df7[feature_cols] = df7[feature_cols].fillna(0)

# 5. Extract and Scale features for the ML model
X = df7[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df7['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Raw Profile Breakdown to help map the text identities
print("\n--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df7.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 7: BOOTH COUNT PER CLUSTER ---")
print(df7['Cluster_ID'].value_counts())

# 8. Export individual target files for campaign ground teams
for cluster_num in range(optimal_k):
    target_cols = [station_col, building_col, area_col, 'Winner_Party', 'Margin_Percentage']
    # Filter only existing tracking columns to prevent target slice export errors
    valid_target_cols = [c for c in target_cols if c in df7.columns]
    
    cluster_df = df7[df7['Cluster_ID'] == cluster_num][valid_target_cols]
    filename = f"Dataset_7_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated for all 4 clusters.")


ValueError: Columns must be same length as key

In [4]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the new dataset (Update the filename to match your file)
df7 = pd.read_csv("Detailed_Polling_Data.csv")

# Standardise column spacing and strip whitespace to prevent key errors
df7.columns = df7.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

# FIX: Drop any existing calculated column copies to prevent structural duplication errors
cols_to_clear = ['Margin_Percentage', 'Winner_Votes', 'Runner_Up_Votes', 'Margin_Of_Victory', 'Winner_Party', 'Cluster_ID']
df7 = df7.drop(columns=[c for c in cols_to_clear if c in df7.columns], errors='ignore')

# 2. Select the primary political party columns for analysis based on your exact schema
core_parties = [
    'All India Anna Dravida Munnetra Kazhagam', 
    'Bahujan Samaj Party', 
    'Naam Tamilar Katchi', 
    'Indian National Congress',
    'Tamilaga Vettri Kazhagam'
]

# Clean missing numerical fields by filling with 0
df7[core_parties] = df7[core_parties].fillna(0)

# Exact structural column definitions from your dataset
station_col = 'Serial No. Of Polling Station'
building_col = 'Location and Name of Building in Which Polling Station Located'
area_col = 'Polling Areas'

# 3. Calculate true total votes for normalization (Core Parties + NOTA)
df7['Total_Calculated_Votes'] = df7[core_parties].sum(axis=1) + df7['NOTA'].fillna(0)

# Filter out empty entries to completely avoid division by zero errors
df7 = df7[df7['Total_Calculated_Votes'] > 0].copy()
# 4. Feature Engineering: Create normalized percentage shares (%) with unique short names
share_cols = []
for party in core_parties:
    # Generates clean, unique initials: AIADMK, BSP, NTK, INC, TVK
    party_label = "".join([word[0].upper() for word in party.split() if word[0].isalpha()])
    col_name = f'{party_label}_share_pct'
    
    if col_name in df7.columns:
        df7 = df7.drop(columns=[col_name])
        
    df7[col_name] = (df7[party] / df7['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)


# Re-calculate fresh strategic voting margin metrics without indexing conflicts
df7['Winner_Votes'] = df7[core_parties].max(axis=1)
sorted_votes = np.sort(df7[core_parties].values, axis=1)
df7['Runner_Up_Votes'] = sorted_votes[:, -2]
df7['Margin_Of_Victory'] = df7['Winner_Votes'] - df7['Runner_Up_Votes']
df7['Winner_Party'] = df7[core_parties].idxmax(axis=1)
df7['Margin_Percentage'] = (df7['Margin_Of_Victory'] / df7['Total_Calculated_Votes']) * 100

# Set up clean target feature tracking array without any independent metrics
feature_cols = share_cols + ['Margin_Percentage']

# Drop or fill edge-case missing numbers inside target features safely using a separate dataframe slice
X = df7[feature_cols].copy().fillna(0)

# 5. Extract and Scale features for the ML model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df7['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Raw Profile Breakdown to help map the text identities
print("\n--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df7.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 7: BOOTH COUNT PER CLUSTER ---")
print(df7['Cluster_ID'].value_counts())

# 8. Export individual target files for campaign ground teams
for cluster_num in range(optimal_k):
    target_cols = [station_col, building_col, area_col, 'Winner_Party', 'Margin_Percentage']
    # Filter only existing tracking columns to prevent target slice export errors
    valid_target_cols = [c for c in target_cols if c in df7.columns]
    
    cluster_df = df7[df7['Cluster_ID'] == cluster_num][valid_target_cols]
    filename = f"Dataset_7_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated for all 4 clusters.")



--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            AIADMK_share_pct  BSP_share_pct  NTK_share_pct  INC_share_pct  \
Cluster_ID                                                                  
0                      30.74           0.27          12.80          30.26   
1                      47.31           0.33           6.39          20.68   
2                      16.38           0.19           5.21          60.75   
3                      31.40           0.44           6.81          27.18   

            TVK_share_pct  Margin_Percentage  
Cluster_ID                                    
0                   25.49               7.34  
1                   24.65              20.79  
2                   17.14              41.14  
3                   33.43               6.72  

--- DATASET 7: BOOTH COUNT PER CLUSTER ---
Cluster_ID
3    156
1     77
0     63
2     25
Name: count, dtype: int64

Success! Campaign target files generated for all 4 clusters.
